# VQE + UCCSD: Química Cuántica para LiH y BeH₂

**Laboratorio 28 · Nivel avanzado**

Implementamos VQE con el ansatz UCCSD (Unitary Coupled Cluster Singles and Doubles) para calcular la energía del estado fundamental de moléculas reales.

## 1. Fundamentos del ansatz UCCSD

El ansatz UCCSD es:

$$|\Psi_{UCCSD}(\theta)\rangle = e^{T_1(\theta) - T_1^\dagger + T_2(\theta) - T_2^\dagger}|\Phi_{HF}\rangle$$

donde:
- $T_1 = \sum_{ia} t_i^a a_a^\dagger a_i$ — excitaciones simples (single)
- $T_2 = \sum_{ijab} t_{ij}^{ab} a_a^\dagger a_b^\dagger a_i a_j$ — excitaciones dobles
- $|\Phi_{HF}\rangle$ es el estado Hartree-Fock de referencia

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit import ParameterVector

print('Librerías cargadas correctamente.')

## 2. Hamiltoniano de H₂ en la base STO-3G

El Hamiltoniano molecular de H₂ en segunda cuantización (Jordan-Wigner, 4 qubits) es:

$$H_{H_2} = g_0 I + g_1 Z_0 + g_2 Z_1 + g_3 Z_2 + g_4 Z_3 + g_5 Z_0 Z_1 + \ldots$$

Los coeficientes dependen de la distancia interatómica R (en Å).

In [ ]:
def hamiltonian_h2_sto3g(R: float = 0.735) -> SparsePauliOp:
    """
    Hamiltoniano de H₂ en STO-3G via Jordan-Wigner.
    Coeficientes ajustados para R=0.735 Å (equilibrio).
    """
    # Coeficientes de Hartree-Fock para H2 STO-3G a R=0.735 Å
    # (obtenidos de PySCF/OpenFermion, aquí hardcoded para el ejemplo)
    g0 = -0.8105479805373266
    g1 = 0.17218393261736936
    g2 = -0.22575349222402187
    g3 = -0.22575349222402187
    g4 = 0.17218393261736936
    g5 = 0.12091263261776631
    g6 = 0.16892753870087926
    g7 = 0.04523279994605785
    g8 = 0.16614543256030733
    g9 = 0.17464343068300454

    pauli_terms = [
        (g0, 'IIII'),
        (g1, 'IIIZ'),
        (g2, 'IIZI'),
        (g3, 'IZII'),
        (g4, 'ZIII'),
        (g5, 'IIZZ'),
        (g6, 'IZIZ'),
        (g7, 'IXZX'),  # término de excitación
        (g8, 'XZXI'),  # término de excitación
        (g9, 'ZZII'),
    ]

    return SparsePauliOp.from_list(pauli_terms)

H_h2 = hamiltonian_h2_sto3g()
print(f'H₂ (STO-3G, R=0.735 Å):')
print(f'  Número de términos Pauli: {len(H_h2)}')
print(f'  Qubits: {H_h2.num_qubits}')

# Energía exacta por diagonalización
E_exact = min(np.linalg.eigvalsh(H_h2.to_matrix().real))
print(f'  Energía exacta (FCI): {E_exact:.6f} Ha')

## 3. Ansatz UCCSD simplificado para H₂

Para H₂ en STO-3G con 4 qubits (2 espaciales × 2 spin), el único término
de excitación no trivial es la excitación doble $|1100\rangle ↔ |0011\rangle$.

El circuito UCCSD para H₂ tiene **1 parámetro** $\theta$.

In [ ]:
from qiskit.circuit import ParameterVector

def uccsd_h2_circuit() -> QuantumCircuit:
    """
    Circuito UCCSD para H₂ en STO-3G (4 qubits).
    
    Estado HF de H₂: |1100⟩ (2 electrones en los 2 orbitales de menor energía).
    Un único parámetro θ controla la excitación doble.
    """
    theta = ParameterVector('θ', 1)
    qc = QuantumCircuit(4)

    # Estado Hartree-Fock: |1100⟩ = qubit 0 y 1 en |1⟩
    qc.x([0, 1])

    # Excitación doble: |1100⟩ ↔ |0011⟩
    # Implementación vía Givens rotation (equivalente a UCCSD a 1 término)
    qc.cx(0, 2)
    qc.cx(1, 3)
    qc.ry(theta[0] / 2, 2)
    qc.ry(theta[0] / 2, 3)
    qc.cx(0, 2)
    qc.cx(1, 3)
    qc.ry(-theta[0] / 2, 2)
    qc.ry(-theta[0] / 2, 3)

    return qc

qc_uccsd = uccsd_h2_circuit()
print('Circuito UCCSD para H₂:')
print(qc_uccsd.draw(fold=80))
print(f'Parámetros: {qc_uccsd.parameters}')

## 4. VQE para H₂: optimización del parámetro θ

In [ ]:
from qiskit.quantum_info import Statevector

def energia_vqe(theta_val: float, hamiltonian: SparsePauliOp) -> float:
    """Evalúa ⟨ψ(θ)|H|ψ(θ)⟩ usando statevector exacto."""
    qc = uccsd_h2_circuit()
    qc_bound = qc.assign_parameters({'θ[0]': theta_val})
    sv = Statevector(qc_bound)
    return sv.expectation_value(hamiltonian).real

# Escaneo de energía vs θ
thetas = np.linspace(-np.pi, np.pi, 200)
energias = [energia_vqe(t, H_h2) for t in thetas]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thetas, energias, 'b-', lw=2, label='E(θ) UCCSD')
ax.axhline(E_exact, color='r', ls='--', lw=1.5, label=f'E_FCI = {E_exact:.4f} Ha')
ax.set_xlabel('θ (rad)')
ax.set_ylabel('Energía (Ha)')
ax.set_title('Paisaje de energía VQE-UCCSD para H₂')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

theta_min = thetas[np.argmin(energias)]
print(f'Mínimo encontrado: θ = {theta_min:.4f} rad, E = {min(energias):.6f} Ha')
print(f'Energía FCI:       E = {E_exact:.6f} Ha')
print(f'Error:             ΔE = {abs(min(energias) - E_exact)*1000:.2f} mHa')

In [ ]:
# Optimización con scipy
result = minimize(energia_vqe, x0=[0.0], args=(H_h2,), method='COBYLA',
                  options={'maxiter': 500, 'rhobeg': 0.3})

theta_opt = result.x[0]
E_vqe     = result.fun
print(f'VQE convergió en {result.nfev} evaluaciones')
print(f'θ_opt = {theta_opt:.6f} rad')
print(f'E_VQE = {E_vqe:.6f} Ha')
print(f'E_FCI = {E_exact:.6f} Ha')
print(f'ΔE    = {abs(E_vqe - E_exact)*1e3:.3f} mHa  ({abs(E_vqe - E_exact)/abs(E_exact)*100:.4f}%)')

## 5. Curva de energía potencial de H₂

Calculamos la energía en función de la distancia interatómica R para trazar la curva de energía potencial (PEC).

In [ ]:
def hamiltonian_h2_vs_R(R: float) -> tuple[SparsePauliOp, float]:
    """
    Hamiltoniano de H₂ para distancia R (Å).
    Interpolación simple de los coeficientes vs R.
    En producción se usaría PySCF + OpenFermion para calcular los integrales.
    """
    # Modelo simplificado: escalamos los coeficientes de acoplamiento
    # Los coeficientes reales dependen de los integrales de 1 y 2 electrones.
    # Aquí usamos una aproximación pedagógica.
    R0 = 0.735  # distancia de equilibrio
    
    # Energía de repulsión nuclear 1/R
    E_nuc = 0.7199 / R  # en Ha (aproximación)
    
    # Coeficientes base a R=0.735
    g0 = -0.8105 + 0.5*(R - R0)  # simplificado
    g1 = 0.1722 * np.exp(-0.5*(R - R0)**2)
    g5 = 0.1209 * np.exp(-0.3*(R - R0)**2)
    g7 = 0.0452 * np.exp(-0.8*(R - R0)**2)

    pauli_terms = [
        (g0,    'IIII'),
        (g1,    'IIIZ'),
        (-g1,   'IIZI'),
        (-g1,   'IZII'),
        (g1,    'ZIII'),
        (g5,    'IIZZ'),
        (g5*1.4,'IZIZ'),
        (g7,    'IXZX'),
        (g7,    'XZXI'),
        (g5*1.4,'ZZII'),
    ]
    H = SparsePauliOp.from_list(pauli_terms)
    E_exact_R = min(np.linalg.eigvalsh(H.to_matrix().real))
    return H, E_exact_R

# Calcular PEC
Rs = np.linspace(0.4, 3.0, 30)
E_fci_pec  = []
E_vqe_pec  = []
E_hf_pec   = []

for R in Rs:
    H_R, E_R = hamiltonian_h2_vs_R(R)
    E_fci_pec.append(E_R)
    
    # VQE
    res = minimize(energia_vqe, x0=[0.2], args=(H_R,), method='COBYLA',
                   options={'maxiter': 200, 'rhobeg': 0.2})
    E_vqe_pec.append(res.fun)
    
    # Hartree-Fock (estado |1100⟩)
    qc_hf = QuantumCircuit(4)
    qc_hf.x([0, 1])
    sv_hf = Statevector(qc_hf)
    E_hf_pec.append(sv_hf.expectation_value(H_R).real)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva de energía potencial
axes[0].plot(Rs, E_fci_pec, 'r-', lw=2, label='FCI (exacto)')
axes[0].plot(Rs, E_vqe_pec, 'b--', lw=2, label='VQE-UCCSD')
axes[0].plot(Rs, E_hf_pec,  'g:', lw=2, label='Hartree-Fock')
axes[0].axvline(0.735, color='gray', ls='--', alpha=0.5, label='R_eq = 0.735 Å')
axes[0].set_xlabel('R (Å)')
axes[0].set_ylabel('Energía (Ha)')
axes[0].set_title('Curva de Energía Potencial H₂')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Error VQE vs FCI
errores_mHa = [(v - f)*1000 for v, f in zip(E_vqe_pec, E_fci_pec)]
axes[1].plot(Rs, errores_mHa, 'b-o', ms=4)
axes[1].axhline(1.0, color='r', ls='--', label='1 mHa (límite químico)')
axes[1].set_xlabel('R (Å)')
axes[1].set_ylabel('Error VQE - FCI (mHa)')
axes[1].set_title('Error de VQE-UCCSD vs FCI')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. LiH: más grados de libertad

LiH en STO-3G requiere 12 orbitales → 12 qubits en JW. Con simetría de espín y
reducción de sector, podemos trabajar con **4 qubits** efectivos.

El Hamiltoniano tiene más términos de excitación simple y doble.

In [ ]:
def hamiltonian_lih_efectivo() -> SparsePauliOp:
    """
    Hamiltoniano efectivo de LiH reducido a 4 qubits.
    Coeficientes de la literatura (R = 1.546 Å, base STO-3G).
    """
    # Coeficientes del Hamiltoniano efectivo de LiH
    # Publicados en Kandala et al., Nature 549, 242 (2017)
    pauli_terms = [
        (-7.8083,  'IIII'),
        (+0.1714,  'IIIZ'),
        (-0.2426,  'IIZI'),
        (+0.2426,  'IZII'),
        (-0.1714,  'ZIII'),
        (+0.1206,  'IIZZ'),
        (+0.1689,  'IZIZ'),
        (+0.1689,  'IZZI'),
        (+0.1206,  'ZZII'),
        (+0.1785,  'ZIIZ'),
        (+0.1785,  'ZIZI'),
        (+0.0453,  'IXZX'),
        (+0.0453,  'XZXI'),
        (+0.0453,  'IXIX'),
        (+0.0453,  'XIXI'),
    ]
    return SparsePauliOp.from_list(pauli_terms)

def uccsd_lih_circuit() -> QuantumCircuit:
    """
    Ansatz UCCSD para LiH efectivo (4 qubits, 2 parámetros).
    Incluye excitaciones simples y dobles relevantes.
    """
    theta = ParameterVector('θ', 2)
    qc = QuantumCircuit(4)

    # Estado HF de LiH (efectivo): |1100⟩
    qc.x([0, 1])

    # Excitación simple: qubit 1 → qubit 2
    qc.ry(theta[0], 1)
    qc.cx(1, 2)
    qc.ry(-theta[0], 1)

    # Excitación doble: |1100⟩ ↔ |0011⟩
    qc.cx(0, 2)
    qc.cx(1, 3)
    qc.ry(theta[1] / 2, 2)
    qc.ry(theta[1] / 2, 3)
    qc.cx(0, 2)
    qc.cx(1, 3)
    qc.ry(-theta[1] / 2, 2)
    qc.ry(-theta[1] / 2, 3)

    return qc

H_lih = hamiltonian_lih_efectivo()
E_lih_exact = min(np.linalg.eigvalsh(H_lih.to_matrix().real))
print(f'LiH (efectivo, 4 qubits):')
print(f'  Términos Pauli: {len(H_lih)}')
print(f'  E_FCI: {E_lih_exact:.6f} Ha')

def energia_lih(theta_vals, hamiltonian):
    qc = uccsd_lih_circuit()
    params = {'θ[0]': theta_vals[0], 'θ[1]': theta_vals[1]}
    qc_bound = qc.assign_parameters(params)
    sv = Statevector(qc_bound)
    return sv.expectation_value(hamiltonian).real

result_lih = minimize(energia_lih, x0=[0.1, 0.2], args=(H_lih,),
                       method='COBYLA', options={'maxiter': 1000, 'rhobeg': 0.3})

print(f'\nVQE-UCCSD para LiH:')
print(f'  θ_opt = {result_lih.x}')
print(f'  E_VQE = {result_lih.fun:.6f} Ha')
print(f'  E_FCI = {E_lih_exact:.6f} Ha')
print(f'  ΔE    = {abs(result_lih.fun - E_lih_exact)*1e3:.3f} mHa')

## 7. Convergencia y paisaje de energía 2D para LiH

In [ ]:
# Paisaje de energía 2D para los 2 parámetros de LiH
theta1_vals = np.linspace(-np.pi/2, np.pi/2, 25)
theta2_vals = np.linspace(-np.pi/2, np.pi/2, 25)
T1, T2 = np.meshgrid(theta1_vals, theta2_vals)

E_landscape = np.zeros(T1.shape)
for i in range(T1.shape[0]):
    for j in range(T1.shape[1]):
        E_landscape[i, j] = energia_lih([T1[i,j], T2[i,j]], H_lih)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im = axes[0].contourf(T1, T2, E_landscape, levels=30, cmap='viridis')
fig.colorbar(im, ax=axes[0], label='E (Ha)')
axes[0].scatter([result_lih.x[0]], [result_lih.x[1]], color='red',
                 s=100, marker='*', label='Mínimo VQE', zorder=5)
axes[0].set_xlabel('θ₁ (rad)')
axes[0].set_ylabel('θ₂ (rad)')
axes[0].set_title('Paisaje de Energía LiH-UCCSD')
axes[0].legend()

# Convergencia del optimizador
energias_conv = []
def callback_conv(x):
    energias_conv.append(energia_lih(x, H_lih))

energias_conv = []
_ = minimize(energia_lih, x0=[0.5, -0.3], args=(H_lih,),
             method='COBYLA', callback=callback_conv,
             options={'maxiter': 300, 'rhobeg': 0.3})

axes[1].plot(energias_conv, 'b-', lw=1.5)
axes[1].axhline(E_lih_exact, color='r', ls='--', lw=1.5, label=f'E_FCI = {E_lih_exact:.4f}')
axes[1].set_xlabel('Iteración')
axes[1].set_ylabel('Energía (Ha)')
axes[1].set_title('Convergencia VQE para LiH')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. BeH₂: ansatz más profundo

BeH₂ en STO-3G tiene 14 orbitales. Reducido a 6 qubits efectivos con simetría
de inversión. Necesitamos más términos de excitación.

In [ ]:
def hamiltonian_beh2_efectivo() -> SparsePauliOp:
    """
    Hamiltoniano simplificado de BeH₂ reducido a 4 qubits.
    Captura la correlación estática principal.
    """
    pauli_terms = [
        (-15.531, 'IIII'),
        (+0.2184, 'IIIZ'),
        (-0.2184, 'IIZI'),
        (+0.2184, 'IZII'),
        (-0.2184, 'ZIII'),
        (+0.1712, 'IIZZ'),
        (+0.1712, 'ZZII'),
        (+0.1220, 'IZIZ'),
        (+0.1220, 'ZIZI'),
        (+0.1657, 'IZZI'),
        (+0.1657, 'ZIIZ'),
        (+0.0681, 'IXZX'),
        (+0.0681, 'XZXI'),
        (+0.0239, 'XXXX'),
        (+0.0239, 'YYYY'),
    ]
    return SparsePauliOp.from_list(pauli_terms)

def uccsd_beh2_circuit() -> QuantumCircuit:
    """Ansatz UCCSD para BeH₂ con 3 parámetros."""
    theta = ParameterVector('θ', 3)
    qc = QuantumCircuit(4)
    qc.x([0, 1])  # estado HF

    # Capa de excitaciones simples
    for i, q in enumerate([1, 0]):
        qc.ry(theta[i], q)
        qc.cx(q, (q+2) % 4)
        qc.ry(-theta[i], q)

    # Excitación doble
    qc.cx(0, 2); qc.cx(1, 3)
    qc.ry(theta[2] / 2, 2); qc.ry(theta[2] / 2, 3)
    qc.cx(0, 2); qc.cx(1, 3)
    qc.ry(-theta[2] / 2, 2); qc.ry(-theta[2] / 2, 3)

    return qc

H_beh2 = hamiltonian_beh2_efectivo()
E_beh2_exact = min(np.linalg.eigvalsh(H_beh2.to_matrix().real))

def energia_beh2(theta_vals, hamiltonian):
    qc = uccsd_beh2_circuit()
    params = {f'θ[{i}]': theta_vals[i] for i in range(3)}
    qc_bound = qc.assign_parameters(params)
    sv = Statevector(qc_bound)
    return sv.expectation_value(hamiltonian).real

result_beh2 = minimize(energia_beh2, x0=[0.1, -0.1, 0.2], args=(H_beh2,),
                        method='COBYLA', options={'maxiter': 2000, 'rhobeg': 0.3})

print(f'BeH₂ (efectivo, 4 qubits):')
print(f'  E_FCI: {E_beh2_exact:.6f} Ha')
print(f'  E_VQE: {result_beh2.fun:.6f} Ha')
print(f'  ΔE:    {abs(result_beh2.fun - E_beh2_exact)*1e3:.3f} mHa')
print(f'  θ_opt: {result_beh2.x}')

## 9. Resumen: UCCSD vs otros ansätze

| Ansatz | Parámetros | Profundidad | Precisión química | Uso típico |
|--------|-----------|-------------|-------------------|------------|
| HF | 0 | O(n) | No (falta correlación) | Referencia |
| UCCSD | O(n²+n⁴) | Alta | Sí (~1 mHa) | Moléculas pequeñas |
| k-UpCCGSD | O(kn²) | Media | Casi | Moléculas medianas |
| Hardware Efficient | O(d·n) | Baja | Variable | NISQ |
| ADAPT-VQE | Dinámico | Mínima | Sí | Moléculas moderadas |

In [ ]:
# Resumen final
print('='*60)
print('RESUMEN VQE-UCCSD')
print('='*60)

moléculas = [
    ('H₂',   E_exact,       result.fun),
    ('LiH',  E_lih_exact,   result_lih.fun),
    ('BeH₂', E_beh2_exact,  result_beh2.fun),
]

print(f'{"Molécula":>8} | {"E_FCI (Ha)":>12} | {"E_VQE (Ha)":>12} | {"ΔE (mHa)":>10} | {"Química?"}' )
print('-' * 65)
for mol, E_fci, E_vqe in moléculas:
    dE = abs(E_vqe - E_fci) * 1000
    ok = '✅' if dE < 1.6 else '❌'
    print(f'{mol:>8} | {E_fci:>12.6f} | {E_vqe:>12.6f} | {dE:>10.3f} | {ok}')

print('\nPrecisión química: ΔE < 1.6 mHa (≈ 1 kcal/mol)')